# Госпитальные_исходы: SHAP окончательной модели

Переносимая версия расчётного кода. Выводы очищены. Результаты записываются только в `runs/vascular`. Исходные снимки в `results/` не изменяются. Полный поиск в 02 требует существенного времени. См. README и docs/REPRODUCIBILITY.md.


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, platform, importlib, datetime, tempfile
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "clinical-mortality-mpl"))
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "repository_support.py").exists())
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from repository_support import initialize
OUT = initialize("vascular")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE_SEED = 2026
VERSIONS = {m: importlib.import_module(m).__version__ for m in ["numpy", "pandas", "sklearn", "optuna", "shap", "catboost", "xgboost", "nbclient"]}
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def savejson(obj, name): (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str))
def figsave(name):
    plt.tight_layout(); plt.savefig(OUT / (name + ".png"), dpi=180, bbox_inches="tight"); plt.savefig(OUT / (name + ".svg"), bbox_inches="tight"); plt.show(); plt.close()
print("Working directory:", OUT, "Python:", platform.python_version(), "seed:", BASE_SEED, VERSIONS)


In [ ]:

import joblib,shap
from sklearn.metrics import average_precision_score,roc_auc_score,brier_score_loss,confusion_matrix
from scipy.stats import beta
from threadpoolctl import threadpool_limits
threadpool_limits(2)
meta=json.loads((OUT/'01_конфигурация.json').read_text());protocol=json.loads((OUT/'02_обновлённые_протокол.json').read_text());final=json.loads((OUT/'02_обновлённые_final_config.json').read_text());provenance=json.loads((OUT/'02_обновлённые_происхождение.json').read_text())
RUN_ID=protocol['run_id'];assert RUN_ID==final['run_id']==provenance['run_id']
assert sha(OUT/'01_данные.csv')==meta['data_sha256']==provenance['data_sha256']
assert sha(OUT/'02_обновлённые_final_model.joblib')==provenance['model_sha256']
D=pd.read_csv(OUT/'01_данные.csv');artifact=joblib.load(OUT/'02_обновлённые_final_model.joblib');model=artifact['model'];pre=artifact['preprocessor'];family=artifact['family']
assert artifact['run_id']==RUN_ID
FEATURES=artifact['features'];X=D[FEATURES];y=D.hospital_death.to_numpy();LABELS=meta['labels']
def transform_pre(pre,a):
 zn=pre['scaler'].transform(pre['num_imputer'].transform(a[pre['nums']]))
 zc=pre['cat_imputer'].transform(a[pre['cats']])
 flags=a[pre['columns']].isna().to_numpy(float) if pre['indicators'] else np.zeros((len(a),0))
 return np.c_[zn,pre['encoder'].transform(zc),flags]
def inclusion_probability(a):
 ages=np.where(a.age_years.isna(),-1,np.searchsorted([60,75],a.age_years.to_numpy(),side='right'))
 keys=[f'{int(v)}_{int(age)}' for v,age in zip(a.vascular_group,ages)]
 return np.array([pre['inclusion_rates'].get(key,1.) for key in keys])
def predict_pipeline(values):
 a=pd.DataFrame(values,columns=FEATURES)
 raw=model.predict_proba(transform_pre(pre,a))[:,1].astype(float)
 r=inclusion_probability(a)
 return r*raw/(1-raw+r*raw)
values=X.to_numpy(float);p=predict_pipeline(values);praw=model.predict_proba(transform_pre(pre,X))[:,1]
assert list(model.classes_)==[0,1] and np.isfinite(p).all()
bgids=np.sort(np.random.default_rng(BASE_SEED).choice(len(D),50,replace=False));bg=values[bgids]
np.savez_compressed(OUT/'03_обновлённые_background.npz',row_id=D.row_id.to_numpy()[bgids],values=bg)
# SHAP explains the complete fixed pipeline INCLUDING the sampling-odds correction.
# Fourteen original predictors are the players; missing indicators are deterministically recomputed.
labels=[LABELS[f] for f in FEATURES]
prediction_jobs=model.n_jobs if family=='RF' else None
if family=='RF':model.set_params(n_jobs=1)
explainer=shap.PermutationExplainer(predict_pipeline,shap.maskers.Independent(bg,max_samples=50),feature_names=labels,seed=BASE_SEED)
varying_counts=np.array([np.count_nonzero(~np.all(explainer.masker.invariants(row),axis=0)) for row in values])
cycles=290//(2*varying_counts+1)
assert cycles.min()>=10
pd.DataFrame(dict(row_id=D.row_id,varying_features=varying_counts,permutation_cycles=cycles,max_evals=290)).to_csv(OUT/'03_обновлённые_SHAP_бюджет.csv',index=False)
explanation=explainer(values,max_evals=10*(2*len(FEATURES)+1),batch_size=50000,silent=True)
if family=='RF':model.set_params(n_jobs=prediction_jobs)
phi=np.asarray(explanation.values);base=np.asarray(explanation.base_values).reshape(-1)
error=float(np.max(abs(base+phi.sum(axis=1)-p)));assert error<1e-5,error
np.savez_compressed(OUT/'03_обновлённые_SHAP.npz',values=phi,base_values=base,p_death=p,p_raw=praw,row_id=D.row_id.to_numpy(),feature_names=np.array(FEATURES),display_data=values,run_id=np.array(RUN_ID),model_sha256=np.array(sha(OUT/'02_обновлённые_final_model.joblib')),data_sha256=np.array(meta['data_sha256']))
shapconfig=dict(run_id=RUN_ID,family=family,method='PermutationExplainer of complete corrected pipeline; max_evals=290; at least 10 permutation cycles; independent masker',scale='probability of hospital death after train-only sampling-odds adjustment',prediction_threads_during_SHAP=1 if family=='RF' else 2,batch_size=50000,background_n=50,background_seed=BASE_SEED,n_explained=len(D),n_retained_training=len(pre['fit_row_ids']),max_evals=10*(2*len(FEATURES)+1),min_permutation_cycles=int(cycles.min()),max_permutation_cycles=int(cycles.max()),max_additivity_error=error,tolerance=1e-5,model_sha256=sha(OUT/'02_обновлённые_final_model.joblib'),versions=VERSIONS,players=FEATURES,missing_indicators='deterministic pipeline components, not separate Shapley players',interpretation='fixed retrospective artifact; explanations not external validation; finite permutations approximate attribution')
savejson(shapconfig,'03_обновлённые_конфигурация_SHAP.json')
savejson(model.get_all_params() if family=='CatBoost' else model.get_params(),'03_обновлённые_параметры_модели.json')
importance=pd.DataFrame(dict(key=FEATURES,label=labels,mean_abs_SHAP=np.mean(abs(phi),axis=0))).sort_values('mean_abs_SHAP',ascending=False)
importance.to_csv(OUT/'03_обновлённые_важности_SHAP.csv',index=False)
shap.plots.bar(explanation,max_display=14,show=False);plt.xlabel('Средний абсолютный SHAP, вероятность смерти');figsave('03_обновлённые_SHAP_bar')
shap.plots.beeswarm(explanation,max_display=14,show=False,plot_size=(11,8));plt.xlabel('SHAP, вероятность смерти');figsave('03_обновлённые_SHAP_beeswarm')
print('Only final model explained',shapconfig);display(importance)


## Зависимости и индивидуальные примеры

In [ ]:

for f in list(dict.fromkeys(importance.key.head(5).tolist()+['vascular_group'])):
 j=FEATURES.index(f);obs=X[f].notna().to_numpy();xx=X.loc[obs,f].to_numpy()
 if f not in meta['continuous']:xx=xx+np.random.default_rng(BASE_SEED).uniform(-.08,.08,sum(obs))
 plt.figure(figsize=(8,5));plt.scatter(xx,phi[obs,j],c=y[obs],cmap='coolwarm',vmin=0,vmax=1,s=18,alpha=.65)
 if (~obs).any():plt.scatter(np.full(sum(~obs),X[f].median()),phi[~obs,j],marker='x',color='grey',label=f'Нет записи, n={sum(~obs)}');plt.legend()
 if f=='vascular_group':plt.xticks([0,1,2],['Периферические','АБШ / аорта','Сонные'])
 plt.xlabel(LABELS[f]);plt.ylabel('SHAP, вероятность смерти');plt.axhline(0,color='grey',lw=.7);plt.title('Окончательный скорректированный пайплайн');figsave('03_обновлённые_зависимость_'+f)
t=artifact['threshold'];pred=p>=t
examples=[('первая смерть',int(np.flatnonzero(y==1)[0])),('первый выживший',int(np.flatnonzero(y==0)[0])),('ближайший к порогу',int(np.argmin(abs(p-t))))]
for title,mask in [('первая FN артефакта',(y==1)&~pred),('первая FP артефакта',(y==0)&pred)]:
 if mask.any():examples.append((title,int(np.flatnonzero(mask)[0])))
def waterfall(i,title):
 vv=phi[i];order=np.argsort(-abs(vv));top=order[:10];rest=order[10:];dd=list(vv[top])+[float(vv[rest].sum())];lab=[]
 for j in top:
  f=FEATURES[j];val=values[i,j];value='нет записи' if np.isnan(val) else format(val,'.3g')
  if f=='vascular_group':value=meta['vascular_labels'][str(int(val))]
  lab.append(value+' | '+LABELS[f])
 lab.append(f'Остальные {len(rest)} признаков');running=float(base[i]);segments=[]
 for j in range(len(dd)-1,-1,-1):
  start=running;running+=dd[j];segments.append((j,start,running))
 assert abs(running-p[i])<1e-5
 traj=[base[i]]+[en for _,_,en in segments];span=max(max(traj)-min(traj),.02)
 fig,(la,ax)=plt.subplots(1,2,figsize=(17,7),sharey=True,gridspec_kw={'width_ratios':[1.7,2]})
 la.set_xlim(0,1);la.set_ylim(len(dd)-.5,-.5);la.axis('off')
 for j,label in enumerate(lab):la.text(.98,j,label,ha='right',va='center',fontsize=10)
 for j,st,en in segments:
  color='#ff0051' if dd[j]>=0 else '#008bfb'
  ax.barh(j,abs(dd[j]),left=min(st,en),height=.64,color=color);ax.text(en+(.015*span if dd[j]>=0 else -.015*span),j,f'{dd[j]:+.4f}',ha='left' if dd[j]>=0 else 'right',va='center',fontsize=9,color=color)
 ax.axvline(base[i],ls='--',color='grey');ax.axvline(p[i],ls='--',color='black');ax.set_xlim(min(traj)-.2*span,max(traj)+.25*span);ax.set_yticks([]);ax.set_xlabel('Вероятностная шкала: baseline + вклады')
 fig.suptitle(f'{title}; row_id={int(D.row_id.iloc[i])}; p={p[i]:.4f}; raw={praw[i]:.4f}\nПрогноз фиксированного артефакта, не внешний OOF');figsave('03_обновлённые_waterfall_'+str(int(D.row_id.iloc[i])))
for i in sorted(set(i for _,i in examples)):waterfall(i,' / '.join(title for title,j in examples if j==i))
pd.DataFrame([dict(reason=title,row_id=int(D.row_id.iloc[i]),y_true=int(y[i]),p_death=float(p[i]),p_raw=float(praw[i]),threshold=t,retained_for_final_train=int(i in pre['fit_row_ids'])) for title,i in examples]).to_csv(OUT/'03_обновлённые_индивидуальные_примеры.csv',index=False)
pd.DataFrame(phi,columns=FEATURES).groupby(D.vascular_group.to_numpy()).mean().to_csv(OUT/'03_обновлённые_SHAP_по_бассейнам.csv')


## Реальные внешние ошибки по бассейнам

In [ ]:

all_oof=pd.read_csv(OUT/'02_обновлённые_oof.csv')
assert set(all_oof.run_id)=={RUN_ID}
oof=all_oof[all_oof.family=='selector'].merge(D[['row_id']+meta['features']],on='row_id',validate='many_to_one')
oof['predicted_death']=oof.p_death>=oof.threshold
oof['error_type']=np.select([(oof.y_true==1)&oof.predicted_death,(oof.y_true==1)&~oof.predicted_death,(oof.y_true==0)&oof.predicted_death],['TP','FN','FP'],default='TN')
oof['missing_count']=oof[meta['features']].isna().sum(axis=1)
oof[['row_id','group_id','repeat','fold','vascular_group','y_true','p_death','p_raw','threshold','error_type','missing_count','configuration']].to_csv(OUT/'03_обновлённые_ошибки.csv',index=False)
def group_metrics(v):
 yy=v.y_true.to_numpy();pp=v.p_death.to_numpy();tn,fp,fn,tp=confusion_matrix(yy,v.predicted_death,labels=[0,1]).ravel()
 return dict(n=len(v),deaths=int(sum(yy)),AP=average_precision_score(yy,pp) if sum(yy) else np.nan,ROC_AUC=roc_auc_score(yy,pp) if len(np.unique(yy))==2 else np.nan,Brier=brier_score_loss(yy,pp),recall_death=tp/(tp+fn) if tp+fn else np.nan,specificity=tn/(tn+fp),precision=tp/(tp+fp) if tp+fp else 0,TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp),mean_prediction=float(pp.mean()),missing_mean=float(v.missing_count.mean()))
gm=[]
for ff in ['LR','RF','CatBoost','XGBoost','selector','selector_all_real']:
 vv=all_oof[all_oof.family==ff].merge(D[['row_id']+meta['features']],on='row_id',validate='many_to_one');vv['predicted_death']=vv.p_death>=vv.threshold;vv['missing_count']=vv[meta['features']].isna().sum(axis=1)
 for (gg,rr),v in vv.groupby(['vascular_group','repeat']):gm.append(dict(family=ff,vascular_group=int(gg),repeat=int(rr),group_label=meta['vascular_labels'][str(gg)],**group_metrics(v)))
gm=pd.DataFrame(gm);gm.to_csv(OUT/'03_обновлённые_качество_по_бассейнам.csv',index=False)
gmetrics=gm[gm.family=='selector']
errors=oof.groupby(['vascular_group','repeat','error_type']).agg(n=('row_id','size'),deaths=('y_true','sum'),mean_p=('p_death','mean'),mean_missing=('missing_count','mean')).reset_index();errors.to_csv(OUT/'03_обновлённые_ошибки_и_полнота.csv',index=False)
intervals=[]
for _,r in gmetrics.iterrows():
 tp=int(r.TP);n=int(r.deaths);lo=0 if tp==0 else beta.ppf(.025,tp,n-tp+1);hi=1 if tp==n else beta.ppf(.975,tp+1,n-tp)
 intervals.append(dict(vascular_group=int(r.vascular_group),repeat=int(r['repeat']),real_deaths=n,TP=tp,recall=r.recall_death,lower=lo,upper=hi,scope='exact binomial descriptive per repeat; no training uncertainty'))
pd.DataFrame(intervals).to_csv(OUT/'03_обновлённые_recall_интервалы.csv',index=False)
fig,axes=plt.subplots(1,3,figsize=(15,4))
for rr,ax in enumerate(axes):
 v=oof[oof.repeat==rr];matrix=confusion_matrix(v.y_true,v.predicted_death,labels=[0,1]);ax.imshow(matrix,cmap='Blues')
 for i in range(2):
  for j in range(2):ax.text(j,i,str(matrix[i,j]),ha='center',va='center',color='white' if matrix[i,j]>matrix.max()/2 else 'black')
 ax.set_xticks([0,1],['Жив','Смерть']);ax.set_yticks([0,1],['Жив','Смерть']);ax.set(xlabel='Прогноз',ylabel='Факт',title=f'Внешний повтор {rr+1}')
figsave('03_обновлённые_матрицы_ошибок')
cal=pd.read_csv(OUT/'02_обновлённые_калибровка.csv')
for fam in ['selector','selector_all_real']:
 fig,(ax,tabax)=plt.subplots(1,2,figsize=(12,5),gridspec_kw={'width_ratios':[1.1,1]});rows=[]
 for r,v in cal[cal.family==fam].groupby('repeat'):
  ax.plot(v.mean_prediction,v.event_rate,'o-',label=f'Повторение {r+1}');rows.extend([r+1,j+1,int(q.n),int(q.deaths),f'{q.mean_prediction:.3f}',f'{q.event_rate:.3f}'] for j,(_,q) in enumerate(v.iterrows()))
 ax.plot([0,1],[0,1],'--',color='grey');ax.set(xlabel='Средний прогноз смерти',ylabel='Наблюдаемая смертность',title=fam);ax.legend();tabax.axis('off');tab=tabax.table(cellText=rows,colLabels=['Повт.','Бин','n','Смертей','p','Частота'],loc='center');tab.auto_set_font_size(False);tab.set_fontsize(8);tab.scale(1,1.35);figsave('03_обновлённые_калибровка_'+fam)
display(gmetrics);display(errors)


# Composition implied by each real-data sampling scheme on the full cohort.
n1=int(D.hospital_death.sum());n0=int((D.hospital_death==0).sum())
survivors=np.array([n0,min(n0,10*n1),min(n0,5*n1)])
dead=np.full(3,n1);xx=np.arange(3)
plt.figure(figsize=(9,5));plt.bar(xx,survivors,label='Выжившие',color='#35658c');plt.bar(xx,dead,bottom=survivors,label='Смерти',color='#be7488')
for j in range(3):plt.text(j,survivors[j]+dead[j]+8,f'n={survivors[j]+dead[j]}\nСмертей {100*dead[j]/(survivors[j]+dead[j]):.2f}%',ha='center')
plt.xticks(xx,['Исходная когорта','1 смерть : 10 выживших','1 смерть : 5 выживших'])
plt.ylabel('Реальные госпитализации');plt.ylim(0,650);plt.legend();plt.title(f'Состав схем отбора: все {int(y.sum())} отметок смерти сохранены')
figsave('03_обновлённые_состав_схем')


fig,axes=plt.subplots(1,3,figsize=(15,5),sharey=True)
for gg,ax in enumerate(axes):
 for hh,color in [(0,'#35658c'),(1,'#be7488')]:
  vv=oof[(oof.vascular_group==gg)&(oof.y_true==hh)]
  jitter=np.random.default_rng(BASE_SEED).uniform(-.12,.12,len(vv))
  ax.scatter(vv['repeat']+jitter,vv.p_death,s=26 if hh else 10,alpha=.9 if hh else .25,color=color,label='Умершие' if hh else 'Выжившие')
 ng=D[D.vascular_group==gg];ax.set_xticks([0,1,2],['1','2','3'])
 ax.set_title(meta['vascular_labels'][str(gg)]+f'\nn={len(ng)}, смертей={int(ng.hospital_death.sum())}')
 ax.set_xlabel('Повторение внешней проверки');ax.legend()
axes[0].set_ylabel('Скорректированный OOF-прогноз смерти')
figsave('03_обновлённые_прогнозы_по_бассейнам')


## Фактические отчёты и материал для статьи

Постобработка исходного локального архива заменена переносимой проверкой `scripts/verify_repository.py`. Сохранённые результаты сравнения и отчёты доступны в `results/` и `docs/`.


## Независимая проверка сохранённых результатов
Второй проход по исходнику, OOF, разбиениям, SQLite, архиву и повторно загруженной модели. Метрики пересчитываются отдельными функциями.

Постобработка исходного локального архива заменена переносимой проверкой `scripts/verify_repository.py`. Сохранённые результаты сравнения и отчёты доступны в `results/` и `docs/`.


## Интерпретация сохранённых и независимо проверенных результатов

Качество обновлённой разметки ограничено. Средние трёх полных OOF-повторений процедуры selector: AP 0,12941, ROC-AUC 0,58054, Brier 0,09332, recall смерти 72,73%, специфичность 47,19%, precision 13,33%. При контроле только по сосудистому бассейну AP 0,13144, ROC-AUC 0,62459 и Brier 0,08780; Brier постоянного контроля 0,09030. По этим точечным оценкам процедура ML не превзошла бассейновый контроль, а её вероятностная ошибка больше постоянного контроля. Это не утверждение статистически доказанного превосходства контроля: парный интервал именно этого сравнения отдельно не рассчитывался. Высокий recall при низкой специфичности и precision не подтверждает пригодность для клинических решений.

Относительно настроенной процедуры all_real сокращённая процедура дала ΔAP +0,01060 с условным 95% интервалом [−0,00649; 0,03690], включающим ноль. Brier уменьшился на 0,00422, условный интервал [−0,00650; −0,00199], но это не означает преимущества над простыми контролями. Финальный XGBoost выбран по заранее принятому внутреннему AP, а не по внешней таблице; внешний результат сохранён без попытки подобрать более выгодное правило после просмотра проверки. Интервалы относятся к фиксированным OOF и не включают всю неопределённость переобучения. Эти результаты не поддерживают утверждение о надёжном клиническом прогнозе всех 55 отметок.

Временная проверка остаётся существенной: у 10 записей год смерти на 1–7 лет позже года операции. Все 55 отметок сохранены по авторскому уточнению, но госпитальный срок событий независимо не подтверждён; переносить результат в статью как проверенный прогноз смерти в пределах индексной госпитализации без клинической сверки нельзя.

In [ ]:
import subprocess
subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "verify_repository.py"), "--cohort", "vascular"], check=True)
